In [49]:
!pip install google-play-scraper pandas

In [50]:
import pandas as pd
from google_play_scraper import reviews, Sort
from datetime import datetime
import os

Set app information

In [51]:
APP_ID = "my.com.tngdigital.ewallet"

RAW_DIR = "../data/raw_data"
CLEAN_DIR = "../data"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

Collect reviews

In [61]:
result, continuation_token = reviews(
    APP_ID,
    lang="en",
    country="my",
    sort=Sort.NEWEST,
    count=20000
)

len(result)

20000

Convert to DataFrame

In [62]:
df = pd.DataFrame(result)

df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,7bdb015a-5670-4177-8379-b7a2f5f7a35b,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,new update really troublesome! remove the impo...,1,0,1.9.4,2026-06-29 21:45:10,"If you need to find something quickly, you can...",2026-06-29 21:50:25,1.9.4
1,995f7b18-1ded-443e-bc8f-6592f7f8bd81,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,good,5,0,None,2026-06-29 21:23:25,None,NaT,None
2,8b79bed1-74c9-42e2-8749-b682baecd60b,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,One of the most horrible UI design I've ever e...,1,0,None,2026-06-29 20:49:33,"If you need to find something quickly, you can...",2026-06-30 15:39:09,None
3,5c58b099-e276-4769-937d-9bb3efcd00b8,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Bloatware. UI front page now focused on sellin...,1,0,1.9.4,2026-06-29 19:03:11,We apologize if you're currently unsatisfied w...,2026-06-30 15:38:19,1.9.4
4,cb9db6de-88d9-4b19-8999-646665ee8e99,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"After update, changing of the UI is still fine...",1,0,1.9.4,2026-06-29 17:22:46,"If you need to find something quickly, you can...",2026-06-29 17:30:35,1.9.4


Select useful columns

In [68]:
df_reviews = df[[
    "reviewId",
    "content",
    "score",
    "thumbsUpCount",
    "at",
]].copy()

df_reviews.head()

,reviewId,content,score,thumbsUpCount,at
0,7bdb015a-5670-4177-8379-b7a2f5f7a35b,new update really troublesome! remove the impo...,1,0,2026-06-29 21:45:10
1,995f7b18-1ded-443e-bc8f-6592f7f8bd81,good,5,0,2026-06-29 21:23:25
2,8b79bed1-74c9-42e2-8749-b682baecd60b,One of the most horrible UI design I've ever e...,1,0,2026-06-29 20:49:33
3,5c58b099-e276-4769-937d-9bb3efcd00b8,Bloatware. UI front page now focused on sellin...,1,0,2026-06-29 19:03:11
4,cb9db6de-88d9-4b19-8999-646665ee8e99,"After update, changing of the UI is still fine...",1,0,2026-06-29 17:22:46


Rename columns

In [69]:
df_reviews = df_reviews.rename(columns={
    "reviewId": "review_id",
    "content": "review_text",
    "score": "rating",
    "thumbsUpCount": "thumbs_up_count",
    "at": "review_datetime"
})

df_reviews.head()

,review_id,review_text,rating,thumbs_up_count,review_datetime
0,7bdb015a-5670-4177-8379-b7a2f5f7a35b,new update really troublesome! remove the impo...,1,0,2026-06-29 21:45:10
1,995f7b18-1ded-443e-bc8f-6592f7f8bd81,good,5,0,2026-06-29 21:23:25
2,8b79bed1-74c9-42e2-8749-b682baecd60b,One of the most horrible UI design I've ever e...,1,0,2026-06-29 20:49:33
3,5c58b099-e276-4769-937d-9bb3efcd00b8,Bloatware. UI front page now focused on sellin...,1,0,2026-06-29 19:03:11
4,cb9db6de-88d9-4b19-8999-646665ee8e99,"After update, changing of the UI is still fine...",1,0,2026-06-29 17:22:46


Add sentiment label from rating

In [70]:
def label_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df_reviews["sentiment_label"] = df_reviews["rating"].apply(label_sentiment)

df_reviews["sentiment_label"].value_counts()

sentiment_label
positive    14014
negative     5226
neutral       760
Name: count, dtype: int64

Basic data checking

In [71]:
print("Total reviews:", len(df_reviews))
print("Missing review text:", df_reviews["review_text"].isna().sum())
print("Duplicate review IDs:", df_reviews["review_id"].duplicated().sum())

df_reviews.info()

Total reviews: 20000
Missing review text: 0
Duplicate review IDs: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   review_id        20000 non-null  object        
 1   review_text      20000 non-null  object        
 2   rating           20000 non-null  int64         
 3   thumbs_up_count  20000 non-null  int64         
 4   review_datetime  20000 non-null  datetime64[ns]
 5   sentiment_label  20000 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 937.6+ KB


Remove missing and duplicate data

In [72]:
df_reviews = df_reviews.dropna(subset=["review_text"])
df_reviews = df_reviews.drop_duplicates(subset=["review_id"])

print("Final reviews:", len(df_reviews))

Final reviews: 20000


Save raw collected data

In [73]:
raw_output_path = f"{RAW_DIR}/touch_n_go_reviews_raw.csv"

df_reviews.to_csv(raw_output_path, index=False, encoding="utf-8-sig")

raw_output_path

'../data/raw_data/touch_n_go_reviews_raw.csv'

Preview final dataset

In [74]:
df_reviews.head(10)

,review_id,review_text,rating,thumbs_up_count,review_datetime,sentiment_label
0,7bdb015a-5670-4177-8379-b7a2f5f7a35b,new update really troublesome! remove the impo...,1,0,2026-06-29 21:45:10,negative
1,995f7b18-1ded-443e-bc8f-6592f7f8bd81,good,5,0,2026-06-29 21:23:25,positive
2,8b79bed1-74c9-42e2-8749-b682baecd60b,One of the most horrible UI design I've ever e...,1,0,2026-06-29 20:49:33,negative
3,5c58b099-e276-4769-937d-9bb3efcd00b8,Bloatware. UI front page now focused on sellin...,1,0,2026-06-29 19:03:11,negative
4,cb9db6de-88d9-4b19-8999-646665ee8e99,"After update, changing of the UI is still fine...",1,0,2026-06-29 17:22:46,negative
5,306fc9f7-dc58-4169-a8e4-d0e99429bbf6,"The name of the app is Touch 'n Go. However, t...",1,0,2026-06-29 17:21:08,negative
6,30a1f37c-64ab-475c-8f6e-64b8468761de,updated version is very usable and have a lot ...,5,0,2026-06-29 17:11:43,positive
7,e394a692-3fbe-4c57-98cd-d9bfa5127758,Stop bombarding people with promotional ads or...,1,0,2026-06-29 16:44:21,negative
8,e1cf671f-99d2-4b8b-ae97-6bc27ed187e3,design is not user friendly. SACK THE BOSSES!!...,1,0,2026-06-29 16:40:12,negative
9,43b59478-0c6b-4382-9837-4ee5e2d470db,Bad UX,1,0,2026-06-29 16:25:36,negative
